# P7 -- Ablation Study: Impact of Class-Weighted Loss
## Notebook 08: `08_ablation_class_weights.ipynb`

**Project:** Optimizing Inference Latency in Enterprise NLP via Task-Specific Knowledge Distillation  
**Group 15 | Section 2241044 | ITER, Siksha 'O' Anusandhan University**

---

### Purpose

This notebook measures the contribution of class-weighted loss to model performance.

The training set has two significant imbalances:
- Sentiment: neutral 4.8x larger than negative
- Urgency: non-urgent 29x larger than urgent

We applied class weights to compensate. But did they actually help?

**Ablation design:**

| Configuration | Class Weights | Everything Else |
|---|---|---|
| Full pipeline (baseline) | YES | Fixed |
| Ablation (this notebook) | NO | Identical |

If removing class weights causes F1 to drop significantly -- weights were essential.  
If F1 stays similar -- their contribution was minor.

**Expected runtime: ~10 minutes on T4 GPU (single seed=42 run).**

---

### Cell 1 -- Restore Session

Loads all required data from Drive. Identical to previous notebooks.

In [1]:
!pip install transformers scikit-learn pandas numpy torch -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import random
import os

BASE = '/content/drive/MyDrive/KD_Project'

train_df  = pd.read_csv(f'{BASE}/train.csv')
val_df    = pd.read_csv(f'{BASE}/val.csv')
test_df   = pd.read_csv(f'{BASE}/test.csv')
pseudo_df = pd.read_csv(f'{BASE}/pseudo_labels_final.csv')

SENTIMENT_MAP = {'negative': 0, 'neutral': 1, 'positive': 2}
URGENCY_MAP   = {'non-urgent': 0, 'urgent': 1}
INV_SENTIMENT = {v: k for k, v in SENTIMENT_MAP.items()}

train_df['sentiment_id']  = train_df['sentiment'].map(SENTIMENT_MAP)
val_df['sentiment_id']    = val_df['sentiment'].map(SENTIMENT_MAP)
test_df['sentiment_id']   = test_df['sentiment'].map(SENTIMENT_MAP)
pseudo_df['sentiment_id'] = pseudo_df['pseudo_sentiment'].map(SENTIMENT_MAP)
pseudo_df['urgency_id']   = pseudo_df['pseudo_urgency'].map(URGENCY_MAP)

pseudo_df = pseudo_df.dropna(subset=['sentiment_id', 'urgency_id']).reset_index(drop=True)
pseudo_df['sentiment_id'] = pseudo_df['sentiment_id'].astype(int)
pseudo_df['urgency_id']   = pseudo_df['urgency_id'].astype(int)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Train (pseudo): {len(pseudo_df)} samples')
print(f'Val           : {len(val_df)} samples')
print(f'Test          : {len(test_df)} samples')
print(f'Device        : {device}')
print(f'GPU           : {torch.cuda.get_device_name(0)}')

Mounted at /content/drive
Train (pseudo): 3876 samples
Val           : 485 samples
Test          : 485 samples
Device        : cuda
GPU           : Tesla T4


### Cell 2 -- Setup: Datasets, Model, Seed

Defines all reusable components. Seed is fixed at 42 for direct comparability
with the P3 seed-42 run which used class weights.

In [2]:
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertModel
from sklearn.utils.class_weight import compute_class_weight
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report
import time

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

class DualTaskDataset(Dataset):
    def __init__(self, texts, sentiment_labels, urgency_labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True,
            max_length=max_length, return_tensors='pt'
        )
        self.sentiment_labels = torch.tensor(sentiment_labels, dtype=torch.long)
        self.urgency_labels   = torch.tensor(urgency_labels,   dtype=torch.long)
    def __len__(self): return len(self.sentiment_labels)
    def __getitem__(self, idx):
        return {
            'input_ids'      : self.encodings['input_ids'][idx],
            'attention_mask' : self.encodings['attention_mask'][idx],
            'sentiment_label': self.sentiment_labels[idx],
            'urgency_label'  : self.urgency_labels[idx]
        }

class SingleTaskDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True,
            max_length=max_length, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids'     : self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'label'         : self.labels[idx]
        }

class DualHeadDistilBERT(nn.Module):
    def __init__(self, num_sentiment=3, num_urgency=2, dropout=0.3):
        super(DualHeadDistilBERT, self).__init__()
        self.distilbert     = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout        = nn.Dropout(dropout)
        self.sentiment_head = nn.Linear(768, num_sentiment)
        self.urgency_head   = nn.Linear(768, num_urgency)
    def forward(self, input_ids, attention_mask):
        outputs    = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        return self.sentiment_head(cls_output), self.urgency_head(cls_output)

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
print('Setup complete. Seed:', SEED)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Setup complete. Seed: 42


### Cell 3 -- Training Function

Single training function that accepts a `use_weights` boolean.

- `use_weights=True`  -- full KD pipeline (matches P3 seed-42 run)
- `use_weights=False` -- ablation (no class weights)

Everything else is identical: same seed, same data, same architecture,
same hyperparameters, same number of epochs.

In [3]:
EPOCHS = 5
ALPHA  = 0.6
BETA   = 0.4

def train_and_evaluate(use_weights: bool, label: str):
    print(f'\n{"="*60}')
    print(f'{label}')
    print(f'Class weights: {use_weights}')
    print(f'{"="*60}')

    set_seed(SEED)

    # DataLoaders
    train_dataset = DualTaskDataset(
        pseudo_df['text'].values,
        pseudo_df['sentiment_id'].values,
        pseudo_df['urgency_id'].values,
        tokenizer
    )
    val_dataset = DualTaskDataset(
        val_df['text'].values,
        val_df['sentiment_id'].values,
        np.zeros(len(val_df), dtype=int),
        tokenizer
    )
    test_dataset = SingleTaskDataset(
        test_df['text'].values,
        test_df['sentiment_id'].values,
        tokenizer
    )
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

    # Model
    model = DualHeadDistilBERT().to(device)

    # Loss -- with or without class weights
    if use_weights:
        s_weights = compute_class_weight('balanced', classes=np.array([0,1,2]), y=pseudo_df['sentiment_id'].values)
        u_weights = compute_class_weight('balanced', classes=np.array([0,1]),   y=pseudo_df['urgency_id'].values)
        s_tensor  = torch.tensor(s_weights, dtype=torch.float).to(device)
        u_tensor  = torch.tensor(u_weights, dtype=torch.float).to(device)
        s_criterion = nn.CrossEntropyLoss(weight=s_tensor)
        u_criterion = nn.CrossEntropyLoss(weight=u_tensor)
        print(f'Sentiment weights: neg={s_weights[0]:.3f}, neu={s_weights[1]:.3f}, pos={s_weights[2]:.3f}')
        print(f'Urgency weights  : non-urgent={u_weights[0]:.3f}, urgent={u_weights[1]:.3f}')
    else:
        s_criterion = nn.CrossEntropyLoss()  # uniform weights
        u_criterion = nn.CrossEntropyLoss()
        print('No class weights applied -- uniform loss.')

    # Optimiser + scheduler
    optimizer   = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = int(0.1 * total_steps),
        num_training_steps = total_steps
    )

    # Training loop
    best_val_f1      = 0.0
    best_model_state = None
    history          = []

    print(f'\n{"Epoch":<8}{"Train Loss":<14}{"Val F1":<12}{"Time"}')
    print('-' * 45)

    for epoch in range(EPOCHS):
        t0 = time.time()
        model.train()
        total_loss = 0

        for batch in train_loader:
            input_ids  = batch['input_ids'].to(device)
            attn_mask  = batch['attention_mask'].to(device)
            s_labels   = batch['sentiment_label'].to(device)
            u_labels   = batch['urgency_label'].to(device)

            optimizer.zero_grad()
            s_logits, u_logits = model(input_ids, attn_mask)
            loss = ALPHA * s_criterion(s_logits, s_labels) + BETA * u_criterion(u_logits, u_labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        # Validation
        model.eval()
        all_preds, all_labels_v = [], []
        with torch.no_grad():
            for batch in val_loader:
                s_logits, _ = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
                all_preds.extend(torch.argmax(s_logits, 1).cpu().numpy())
                all_labels_v.extend(batch['sentiment_label'].numpy())

        val_f1 = f1_score(all_labels_v, all_preds, average='macro')
        print(f'Epoch {epoch+1:<4} {total_loss/len(train_loader):<14.4f}{val_f1:<12.4f}{time.time()-t0:.1f}s')

        history.append({'epoch': epoch+1, 'train_loss': round(total_loss/len(train_loader),4), 'val_f1': round(val_f1,4)})

        if val_f1 > best_val_f1:
            best_val_f1      = val_f1
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            print(f'         -> Best saved (Val F1: {val_f1:.4f})')

    # Test evaluation
    model.load_state_dict(best_model_state)
    model.eval()
    all_preds, all_labels_t = [], []

    with torch.no_grad():
        for batch in test_loader:
            logits, _ = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
            all_labels_t.extend(batch['label'].numpy())

    macro_f1     = f1_score(all_labels_t, all_preds, average='macro')
    per_class_f1 = f1_score(all_labels_t, all_preds, average=None)
    accuracy     = (np.array(all_preds) == np.array(all_labels_t)).mean()

    print(f'\nTest Macro F1 : {macro_f1:.4f}')
    print(f'Test Accuracy : {accuracy:.4f}')
    print(classification_report(all_labels_t, all_preds, target_names=['negative', 'neutral', 'positive']))

    return {
        'config'      : label,
        'use_weights' : use_weights,
        'macro_f1'    : round(macro_f1, 4),
        'accuracy'    : round(accuracy, 4),
        'negative_f1' : round(per_class_f1[0], 4),
        'neutral_f1'  : round(per_class_f1[1], 4),
        'positive_f1' : round(per_class_f1[2], 4),
        'best_val_f1' : round(best_val_f1, 4),
        'history'     : history
    }

print('Training function defined.')

Training function defined.


### Cell 4 -- Run Full Pipeline (With Class Weights)

This reproduces the P3 seed-42 run exactly.
Result should match: Test Macro F1 = 0.7582 (from P3).

This is the **control** condition.

In [4]:
result_with_weights = train_and_evaluate(
    use_weights = True,
    label       = 'KD PIPELINE -- WITH CLASS WEIGHTS (Control, Seed 42)'
)


KD PIPELINE -- WITH CLASS WEIGHTS (Control, Seed 42)
Class weights: True


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentiment weights: neg=2.584, neu=0.518, pos=1.467
Urgency weights  : non-urgent=0.517, urgent=15.023

Epoch   Train Loss    Val F1      Time
---------------------------------------------
Epoch 1    0.6887        0.7062      43.5s
         -> Best saved (Val F1: 0.7062)
Epoch 2    0.3231        0.7704      45.6s
         -> Best saved (Val F1: 0.7704)
Epoch 3    0.1801        0.7804      49.5s
         -> Best saved (Val F1: 0.7804)
Epoch 4    0.0990        0.7956      48.7s
         -> Best saved (Val F1: 0.7956)
Epoch 5    0.0578        0.7886      49.2s

Test Macro F1 : 0.7670
Test Accuracy : 0.7979
              precision    recall  f1-score   support

    negative       0.72      0.85      0.78        61
     neutral       0.82      0.89      0.85       288
    positive       0.80      0.57      0.67       136

    accuracy                           0.80       485
   macro avg       0.78      0.77      0.77       485
weighted avg       0.80      0.80      0.79       485



### Cell 5 -- Run Ablation (Without Class Weights)

Identical in every way except class weights are removed.
This is the **ablation** condition.

Watch specifically:
- Does negative F1 drop? (604 samples, 12.5% of training set)
- Does positive F1 drop? (1,363 samples, 28.1% of training set)
- Does neutral F1 rise? (2,879 samples, 59.4% -- model may over-predict neutral)

In [5]:
result_no_weights = train_and_evaluate(
    use_weights = False,
    label       = 'KD PIPELINE -- WITHOUT CLASS WEIGHTS (Ablation, Seed 42)'
)


KD PIPELINE -- WITHOUT CLASS WEIGHTS (Ablation, Seed 42)
Class weights: False


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


No class weights applied -- uniform loss.

Epoch   Train Loss    Val F1      Time
---------------------------------------------
Epoch 1    0.4910        0.7540      49.8s
         -> Best saved (Val F1: 0.7540)
Epoch 2    0.2321        0.7793      48.5s
         -> Best saved (Val F1: 0.7793)
Epoch 3    0.1337        0.8001      49.3s
         -> Best saved (Val F1: 0.8001)
Epoch 4    0.0705        0.8201      48.9s
         -> Best saved (Val F1: 0.8201)
Epoch 5    0.0419        0.8090      49.2s

Test Macro F1 : 0.7520
Test Accuracy : 0.7876
              precision    recall  f1-score   support

    negative       0.75      0.75      0.75        61
     neutral       0.79      0.90      0.84       288
    positive       0.79      0.57      0.66       136

    accuracy                           0.79       485
   macro avg       0.78      0.74      0.75       485
weighted avg       0.79      0.79      0.78       485



### Cell 6 -- Direct Comparison Table

Side-by-side comparison of the two configurations.

The difference row shows the exact contribution of class-weighted loss to each metric.

In [6]:
print('=' * 70)
print('ABLATION STUDY -- CLASS WEIGHTS vs NO CLASS WEIGHTS')
print(f'Seed: 42 | Epochs: 5 | lr: 2e-5 | batch: 16')
print('=' * 70)
print(f'{"Metric":<20} {"With Weights":<22} {"No Weights":<22} {"Difference"}')
print('-' * 70)

metrics = [
    ('Macro F1',    'macro_f1'),
    ('Accuracy',    'accuracy'),
    ('Negative F1', 'negative_f1'),
    ('Neutral F1',  'neutral_f1'),
    ('Positive F1', 'positive_f1'),
]

for label, key in metrics:
    with_val = result_with_weights[key]
    no_val   = result_no_weights[key]
    diff     = with_val - no_val
    marker   = ' (+weights better)' if diff > 0.01 else (' (-weights better)' if diff < -0.01 else ' (similar)')
    print(f'{label:<20} {with_val:<22} {no_val:<22} {diff:+.4f}{marker}')

print('=' * 70)

# Overall verdict
macro_diff = result_with_weights['macro_f1'] - result_no_weights['macro_f1']
neg_diff   = result_with_weights['negative_f1'] - result_no_weights['negative_f1']
pos_diff   = result_with_weights['positive_f1'] - result_no_weights['positive_f1']
neu_diff   = result_with_weights['neutral_f1']  - result_no_weights['neutral_f1']

print(f'\nVERDICT:')
if macro_diff > 0.02:
    print(f'Class weights improved Macro F1 by {macro_diff:+.4f} -- SIGNIFICANT contribution.')
elif macro_diff > 0:
    print(f'Class weights improved Macro F1 by {macro_diff:+.4f} -- MODERATE contribution.')
elif macro_diff > -0.01:
    print(f'Macro F1 difference is {macro_diff:+.4f} -- MINOR contribution overall.')
else:
    print(f'No weights outperformed weighted by {abs(macro_diff):.4f} -- UNEXPECTED result.')

print(f'Minority class impact: Negative F1 {neg_diff:+.4f} | Positive F1 {pos_diff:+.4f}')
print(f'Majority class impact: Neutral F1 {neu_diff:+.4f}')

ABLATION STUDY -- CLASS WEIGHTS vs NO CLASS WEIGHTS
Seed: 42 | Epochs: 5 | lr: 2e-5 | batch: 16
Metric               With Weights           No Weights             Difference
----------------------------------------------------------------------
Macro F1             0.767                  0.752                  +0.0150 (+weights better)
Accuracy             0.7979                 0.7876                 +0.0103 (+weights better)
Negative F1          0.782                  0.7541                 +0.0279 (+weights better)
Neutral F1           0.8524                 0.8436                 +0.0088 (similar)
Positive F1          0.6667                 0.6581                 +0.0086 (similar)

VERDICT:
Class weights improved Macro F1 by +0.0150 -- MODERATE contribution.
Minority class impact: Negative F1 +0.0279 | Positive F1 +0.0086
Majority class impact: Neutral F1 +0.0088


### Cell 7 -- Save Results and Commit

Saves the ablation comparison to Drive and prints the GitHub commit command.

In [8]:
# Build results dataframe
rows = []
for label, key in [
    ('macro_f1', 'macro_f1'), ('accuracy', 'accuracy'),
    ('negative_f1', 'negative_f1'), ('neutral_f1', 'neutral_f1'),
    ('positive_f1', 'positive_f1')
]:
    rows.append({
        'metric'          : label,
        'with_weights'    : result_with_weights[key],
        'without_weights' : result_no_weights[key],
        'difference'      : round(result_with_weights[key] - result_no_weights[key], 4)
    })

ablation_df = pd.DataFrame(rows)
ablation_df.to_csv(f'{BASE}/ablation_class_weights.csv', index=False)

# Save training histories
hist_with = pd.DataFrame(result_with_weights['history'])
hist_with['config'] = 'with_weights'
hist_no   = pd.DataFrame(result_no_weights['history'])
hist_no['config'] = 'no_weights'
pd.concat([hist_with, hist_no]).to_csv(f'{BASE}/ablation_training_history.csv', index=False)

macro_diff = result_with_weights['macro_f1'] - result_no_weights['macro_f1']

print('Files saved:')
print(f'  {BASE}/ablation_class_weights.csv')
print(f'  {BASE}/ablation_training_history.csv')

Files saved:
  /content/drive/MyDrive/KD_Project/ablation_class_weights.csv
  /content/drive/MyDrive/KD_Project/ablation_training_history.csv
